# F3 - Notebook 03: Analizador Fitness y POO exploratoria

## Objetivo de este notebook

Este notebook contiene una implementación exploratoria de Programación Orientada a Objetos para la Fase 3.

La pauta no exige una arquitectura POO completa, pero permite incorporar clases simples si corresponde al proyecto.

## Qué se implementa

Una clase llamada:

`AnalizadorFitness`

Esta clase agrupa operaciones frecuentes sobre el dataset procesado:

- Resumen básico.
- Filtrado por región.
- Filtrado por periodo.
- Top de países según variable.
- Promedio por región.

## Resultado esperado

Se espera demostrar cómo una clase simple puede mejorar la organización del código y proyectar una arquitectura más modular.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

## 1. Carga del dataset procesado

Se usa el mismo dataset procesado de Fase 2.

## Resultado esperado

Dataset cargado correctamente desde:

`data/processed/gym_data_processed.csv`

In [2]:
# Busca automáticamente la raíz del proyecto subiendo carpetas
def encontrar_raiz_proyecto(nombre_repo="gym-fitness-analytics"):
    ruta_actual = Path.cwd().resolve()

    for ruta in [ruta_actual] + list(ruta_actual.parents):
        if ruta.name == nombre_repo:
            return ruta

    raise FileNotFoundError(
        f"No se pudo encontrar la raíz del proyecto '{nombre_repo}' desde {ruta_actual}"
    )


PROJECT_ROOT = encontrar_raiz_proyecto()

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "gym_data_processed.csv"

print("Directorio actual:", Path.cwd().resolve())
print("Raíz del proyecto:", PROJECT_ROOT)
print("Ruta del dataset procesado:", DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset procesado en la ruta: {DATA_PATH}"
    )

df = pd.read_csv(DATA_PATH)

print("Dataset procesado cargado correctamente.")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

df.head()

Directorio actual: C:\Users\warevalo\Documents\GitHub\gym-fitness-analytics\src\fase3
Raíz del proyecto: C:\Users\warevalo\Documents\GitHub\gym-fitness-analytics
Ruta del dataset procesado: C:\Users\warevalo\Documents\GitHub\gym-fitness-analytics\data\processed\gym_data_processed.csv
Dataset procesado cargado correctamente.
Filas: 3564
Columnas: 27


,country,year,region,gym_memberships,fitness_participation_rate,total_health_club_revenue_usd,number_of_gyms,gym_penetration_rate,urban_population_percentage,obesity_rate,...,periodo,gym_memberships_norm,total_health_club_revenue_usd_norm,number_of_gyms_norm,gdp_per_capita_usd_norm,population_total_norm,average_membership_cost_usd_norm,memberships_per_100k_norm,gyms_per_100k_norm,revenue_per_membership_usd_norm
0,Angola,2000,Africa,95521.0,0.3873,5731259.0,204.0,0.0059,0.5051,0.0470,...,pre_covid,0.000909,0.000066,0.003230,0.003299,0.011038,0.0,0.012043,0.034128,0.000001
1,Angola,2001,Africa,103840.0,0.3939,6230372.0,222.0,0.0062,0.5172,0.0496,...,pre_covid,0.000990,0.000071,0.003519,0.003080,0.011419,0.0,0.013051,0.036453,0.000001
2,Angola,2002,Africa,121093.0,0.4003,7265583.0,249.0,0.0070,0.5289,0.0522,...,pre_covid,0.001158,0.000084,0.003953,0.006461,0.011819,0.0,0.015678,0.040381,0.000001
3,Angola,2003,Africa,142783.0,0.4065,8566966.0,281.0,0.0080,0.5400,0.0548,...,pre_covid,0.001369,0.000099,0.004467,0.007438,0.012243,0.0,0.018908,0.044930,0.000001
4,Angola,2004,Africa,179615.0,0.4124,10776918.0,325.0,0.0097,0.5504,0.0574,...,pre_covid,0.001728,0.000125,0.005174,0.009747,0.012696,0.0,0.024573,0.051321,0.000001


## 2. Preparación mínima para la clase

Se seleccionan columnas necesarias para el análisis.

## Resultado esperado

Un DataFrame llamado `df_f3` listo para ser usado por la clase.

In [3]:
columnas_f3 = [
    "country",
    "region",
    "year",
    "gym_penetration_rate",
    "memberships_per_100k",
    "gyms_per_100k",
    "revenue_per_membership_usd",
    "periodo"
]

faltantes = [col for col in columnas_f3 if col not in df.columns]

if faltantes:
    raise ValueError(f"Faltan columnas necesarias: {faltantes}")

df_f3 = df[columnas_f3].copy()

df_f3 = df_f3.dropna(
    subset=[
        "gym_penetration_rate",
        "memberships_per_100k",
        "gyms_per_100k"
    ]
)

print("Dataset preparado para AnalizadorFitness.")
print("Filas:", df_f3.shape[0])
print("Columnas:", df_f3.shape[1])

Dataset preparado para AnalizadorFitness.
Filas: 3564
Columnas: 8


# 3. Diseño de la clase `AnalizadorFitness`

## Por qué usar una clase

La clase permite agrupar operaciones relacionadas con el análisis del dataset fitness.

En vez de tener funciones sueltas, la clase mantiene un DataFrame interno:

`self.df`

y ofrece métodos para consultarlo.

## Qué no se implementa

No se implementan:

- Herencia.
- Polimorfismo.
- Clases abstractas.
- Patrones de diseño.

Esto se debe a que la pauta solo sugiere POO si corresponde y no exige arquitectura avanzada.

## Resultado esperado

La clase debe poder instanciarse con el dataset procesado y ejecutar métodos de análisis.

In [4]:
class AnalizadorFitness:
    """
    Clase simple para agrupar operaciones frecuentes sobre el dataset.

    Esta clase no implementa herencia, polimorfismo ni patrones avanzados.
    Su objetivo es demostrar una aproximación inicial a POO y organización
    de responsabilidades.
    """

    def __init__(self, dataframe):
        """
        Inicializa el analizador con un DataFrame.

        Parámetros:
            dataframe: Dataset procesado de Fase 2.

        Resultado esperado:
            Se crea una copia interna del DataFrame en self.df.
        """
        if not isinstance(dataframe, pd.DataFrame):
            raise TypeError("El parámetro dataframe debe ser un pd.DataFrame.")

        self.df = dataframe.copy()

    def resumen_basico(self):
        """
        Entrega un resumen básico del dataset cargado.

        Resultado esperado:
            Diccionario con filas, columnas, países, regiones y años.
        """
        resumen = {
            "filas": self.df.shape[0],
            "columnas": self.df.shape[1]
        }

        if "country" in self.df.columns:
            resumen["paises"] = self.df["country"].nunique()

        if "region" in self.df.columns:
            resumen["regiones"] = self.df["region"].nunique()

        if "year" in self.df.columns and self.df["year"].notna().any():
            resumen["anio_min"] = int(self.df["year"].min())
            resumen["anio_max"] = int(self.df["year"].max())

        return resumen

    def filtrar_por_region(self, region):
        """
        Filtra registros por región.

        Resultado esperado:
            DataFrame con registros de la región solicitada.
        """
        if "region" not in self.df.columns:
            raise ValueError("La columna 'region' no existe.")

        return self.df[self.df["region"] == region].copy()

    def filtrar_por_periodo(self, periodo):
        """
        Filtra registros por periodo.

        Periodos esperados:
            pre_covid, covid, post_covid

        Resultado esperado:
            DataFrame filtrado por periodo.
        """
        if "periodo" not in self.df.columns:
            raise ValueError("La columna 'periodo' no existe.")

        periodos_validos = {"pre_covid", "covid", "post_covid"}

        if periodo not in periodos_validos:
            raise ValueError(
                f"Periodo inválido: {periodo}. Use uno de: {periodos_validos}"
            )

        return self.df[self.df["periodo"] == periodo].copy()

    def obtener_top_paises(self, columna, top_n=10):
        """
        Obtiene los primeros países según una columna numérica.

        Resultado esperado:
            DataFrame ordenado de mayor a menor.
        """
        if columna not in self.df.columns:
            raise ValueError(f"La columna '{columna}' no existe.")

        columnas_salida = ["country", "region", "year", columna]
        faltantes = [col for col in columnas_salida if col not in self.df.columns]

        if faltantes:
            raise ValueError(f"Faltan columnas requeridas: {faltantes}")

        return (
            self.df[columnas_salida]
            .dropna(subset=[columna])
            .sort_values(by=columna, ascending=False)
            .head(top_n)
            .copy()
        )

    def promedio_por_region(self, columna):
        """
        Calcula promedio de una columna por región.

        Resultado esperado:
            Serie de Pandas con promedio por región.
        """
        if "region" not in self.df.columns:
            raise ValueError("La columna 'region' no existe.")

        if columna not in self.df.columns:
            raise ValueError(f"La columna '{columna}' no existe.")

        return self.df.groupby("region")[columna].mean().sort_values(
            ascending=False
        )

## 4. Instanciar la clase

Aquí se crea un objeto llamado `analizador`.

## Resultado esperado

El objeto debe crearse sin errores y quedar listo para ejecutar métodos.

In [5]:
analizador = AnalizadorFitness(df_f3)

print("Objeto AnalizadorFitness creado correctamente.")

Objeto AnalizadorFitness creado correctamente.


## 5. Ejecutar resumen básico

Este método permite verificar rápidamente las características generales del dataset.

## Resultado esperado

Un diccionario con:

- Filas.
- Columnas.
- Número de países.
- Número de regiones.
- Año mínimo.
- Año máximo.

In [6]:
resumen = analizador.resumen_basico()

resumen

{'filas': 3564,
 'columnas': 8,
 'paises': 132,
 'regiones': 6,
 'anio_min': 2000,
 'anio_max': 2026}

## 6. Obtener top de países

Se usa el método `obtener_top_paises()` para obtener los países o registros con mayor valor en una columna.

## Resultado esperado

Una tabla ordenada de mayor a menor según `memberships_per_100k`.

In [7]:
top_paises = analizador.obtener_top_paises(
    columna="memberships_per_100k",
    top_n=10
)

top_paises

,country,region,year,memberships_per_100k
3374,United States of America,North America,2026,30221.918617
3373,United States of America,North America,2025,30000.846077
2429,Netherlands,Europe,2026,29194.416757
2453,Norway,Europe,2023,28987.602546
2428,Netherlands,Europe,2025,28980.856127
2427,Netherlands,Europe,2024,28731.432516
2456,Norway,Europe,2026,28597.186178
2455,Norway,Europe,2025,28388.007851
2454,Norway,Europe,2024,28143.691298
161,Australia,Oceania,2026,27699.213423


## 7. Filtrar por periodo

Se usa el método `filtrar_por_periodo()` para obtener registros de un periodo específico.

## Resultado esperado

Un DataFrame con registros del periodo `post_covid`.

In [8]:
post_covid = analizador.filtrar_por_periodo("post_covid")

print("Registros post_covid:", post_covid.shape[0])
post_covid.head()

Registros post_covid: 660


,country,region,year,gym_penetration_rate,memberships_per_100k,gyms_per_100k,revenue_per_membership_usd,periodo
22,Angola,Africa,2022,0.0304,3042.110615,4.237404,60.719986,post_covid
23,Angola,Africa,2023,0.0295,2948.978972,4.359195,59.999977,post_covid
24,Angola,Africa,2024,0.0290,2895.851694,4.384223,60.000026,post_covid
25,Angola,Africa,2025,0.0292,2920.993007,4.423815,59.999990,post_covid
26,Angola,Africa,2026,0.0294,2942.518195,4.455489,59.999975,post_covid


## 8. Filtrar por región

Se obtiene una región existente desde el dataset y luego se filtran sus registros.

## Resultado esperado

Un DataFrame con registros de la región seleccionada.

In [9]:
region_ejemplo = df_f3["region"].dropna().iloc[0]

registros_region = analizador.filtrar_por_region(region_ejemplo)

print("Región usada como ejemplo:", region_ejemplo)
print("Registros encontrados:", registros_region.shape[0])

registros_region.head()

Región usada como ejemplo: Africa
Registros encontrados: 783


,country,region,year,gym_penetration_rate,memberships_per_100k,gyms_per_100k,revenue_per_membership_usd,periodo
0,Angola,Africa,2000,0.0059,589.822616,1.259658,59.999990,pre_covid
1,Angola,Africa,2001,0.0062,620.043651,1.325594,59.999730,pre_covid
2,Angola,Africa,2002,0.0070,698.840625,1.437006,60.000025,pre_covid
3,Angola,Africa,2003,0.0080,795.727216,1.566008,59.999902,pre_covid
4,Angola,Africa,2004,0.0097,965.650082,1.747272,60.000100,pre_covid


## 9. Promedio por región

Se calcula el promedio de `gym_penetration_rate` agrupado por región.

## Resultado esperado

Una Serie de Pandas con regiones y sus promedios.

In [10]:
promedio_region = analizador.promedio_por_region("gym_penetration_rate")

promedio_region

region
Europe           0.113839
Oceania          0.098022
North America    0.088760
South America    0.055014
Asia             0.050762
Africa           0.013010
Name: gym_penetration_rate, dtype: float64

## 10. Conclusión del notebook POO

La clase `AnalizadorFitness` permite agrupar operaciones frecuentes del proyecto.

Esta implementación ayuda a demostrar:

- Organización de responsabilidades.
- Encapsulamiento básico del DataFrame.
- Reutilización de operaciones.
- Posible evolución modular para fases futuras.

Sin embargo, se mantiene simple porque la Fase 3 no exige arquitectura orientada a objetos completa.